# LeetCode 3901 — Good Subsequence Queries bằng Python

- Link: https://leetcode.com/problems/good-subsequence-queries/
- Độ khó: **Hard**
- Chủ đề: GCD, segment tree (cập nhật điểm, truy vấn đoạn), chia căn (sqrt decomposition), tiền tố/hậu tố GCD, lập luận số học (số nguyên tố đôi một nguyên tố cùng nhau)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countGoodSubseq(self, nums: list[int], p: int, queries: list[list[int]]) -> int:
```

## 1. Mục tiêu học

1. Biến câu hỏi "có tồn tại dãy con có GCD đúng bằng `p`" thành điều kiện trên **GCD của tất cả bội số của `p`**.
2. Xử lý điều kiện phụ "độ dài **nhỏ hơn `n`**" bằng **bỏ một phần tử** và GCD tiền tố/hậu tố.
3. Chứng minh bổ đề số học: nếu `n ≥ 7`, mọi phần tử là bội của `p`, GCD chung là `p` (giá trị ≤ `5·10^4`) thì **luôn bỏ được một phần tử** mà GCD vẫn là `p`.
4. Cài đặt **segment tree GCD** (dạng lặp, nhanh trong Python) để cập nhật điểm và lấy GCD toàn cục / GCD đoạn.
5. So sánh với **chia căn** và kiểm thử với vét cạn.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng `nums` độ dài `n` và số `p`. Một dãy con **khác rỗng** là **tốt** nếu:

- độ dài của nó **nhỏ hơn hẳn `n`** (không được lấy cả mảng);
- GCD các phần tử của nó **đúng bằng `p`**.

Có `q` truy vấn `[ind, val]`: gán `nums[ind] = val`. Sau **mỗi** truy vấn, kiểm tra mảng hiện tại có dãy con tốt hay không. Trả về **số truy vấn** mà câu trả lời là "có".

**Ví dụ 1:** `nums = [4,8,12,16], p = 2, queries = [[0,3],[2,6]]` → `1`. Sau truy vấn 2 mảng là `[3,8,6,16]`, dãy con `[8,6]` có GCD 2.

**Ví dụ 2:** `nums = [4,5,7,8], p = 3, queries = [[0,6],[1,9],[2,3]]` → `2`.

**Ví dụ 3:** `nums = [5,7,9], p = 2, queries = [[1,4],[2,8]]` → `0`.

**Ràng buộc:** `2 ≤ n ≤ 5·10^4`, `1 ≤ nums[i], val, p ≤ 5·10^4`, `1 ≤ q ≤ 5·10^4`.

In [1]:
from math import gcd
from functools import reduce
from itertools import combinations
import random, time

examples = [
    ([4, 8, 12, 16], 2, [[0, 3], [2, 6]], 1),
    ([4, 5, 7, 8], 3, [[0, 6], [1, 9], [2, 3]], 2),
    ([5, 7, 9], 2, [[1, 4], [2, 8]], 0),
]
for nums, p, qs, exp in examples:
    print(nums, "p =", p, qs, "-> kỳ vọng", exp)

[4, 8, 12, 16] p = 2 [[0, 3], [2, 6]] -> kỳ vọng 1
[4, 5, 7, 8] p = 3 [[0, 6], [1, 9], [2, 3]] -> kỳ vọng 2
[5, 7, 9] p = 2 [[1, 4], [2, 8]] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. GCD và tính chất "càng thêm phần tử, GCD càng nhỏ (theo quan hệ chia hết)"

- `gcd(a, 0) = a` ⇒ `0` là **phần tử trung hòa**: tiện dùng cho vị trí "không tham gia".
- `gcd(S ∪ T)` **chia hết** `gcd(S)`. Thêm phần tử chỉ làm GCD "nhỏ đi" (về quan hệ chia hết).
- `functools.reduce(gcd, list, 0)` cho GCD cả danh sách.

In [2]:
print(gcd(12, 0), gcd(0, 0), reduce(gcd, [12, 18, 30], 0), reduce(gcd, [], 0))

12 0 6 0


### 3.2. GCD tiền tố và hậu tố — "bỏ một phần tử"

`pre[i] = gcd(a[0..i-1])`, `suf[i] = gcd(a[i..n-1])`. GCD của mảng khi bỏ phần tử `i` là `gcd(pre[i], suf[i+1])` — tính mọi `i` trong `O(n)`.

In [3]:
a = [6, 10, 15]
n = len(a)
pre = [0] * (n + 1); suf = [0] * (n + 1)
for i in range(n): pre[i + 1] = gcd(pre[i], a[i])
for i in range(n - 1, -1, -1): suf[i] = gcd(suf[i + 1], a[i])
print("GCD cả mảng:", pre[n], "| bỏ từng phần tử:", [gcd(pre[i], suf[i + 1]) for i in range(n)])

GCD cả mảng: 1 | bỏ từng phần tử: [5, 3, 2]


### 3.3. Segment tree dạng lặp cho GCD

Mảng `t` kích thước `2·size`; lá ở `t[size + i]`; nút trong `t[v] = gcd(t[2v], t[2v+1])`. Cập nhật điểm: sửa lá rồi đi lên. Truy vấn `[l, r)`: hai con trỏ đi lên từ hai đầu. Giá trị toàn cục là `t[1]`.

In [4]:
class SegGCD:
    def __init__(self, arr):
        self.size = 1
        while self.size < len(arr):
            self.size *= 2
        self.t = [0] * (2 * self.size)
        self.t[self.size:self.size + len(arr)] = arr
        for v in range(self.size - 1, 0, -1):
            self.t[v] = gcd(self.t[2 * v], self.t[2 * v + 1])
    def update(self, i, val):
        v = i + self.size
        self.t[v] = val
        v //= 2
        while v:
            self.t[v] = gcd(self.t[2 * v], self.t[2 * v + 1])
            v //= 2
    def query(self, l, r):          # gcd trên [l, r)
        res = 0
        l += self.size; r += self.size
        while l < r:
            if l & 1: res = gcd(res, self.t[l]); l += 1
            if r & 1: r -= 1; res = gcd(res, self.t[r])
            l //= 2; r //= 2
        return res
    def total(self):
        return self.t[1]

st = SegGCD([12, 18, 0, 30])
print(st.total(), st.query(0, 2), st.query(2, 4))
st.update(1, 7)
print(st.total())

6 6 30
1


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

**Ai có thể nằm trong dãy con có GCD = `p`?** Mọi phần tử của dãy đó phải **chia hết cho `p`**. Phần tử không chia hết cho `p` thì vô dụng.

Gọi `M` = tập các phần tử là bội của `p`, `g = gcd(M)` (`g = 0` nếu `M` rỗng).

- Mọi dãy con của `M` có GCD là **bội của `g`**. Nếu `g ≠ p` (tức `g = 0` hoặc `g` là bội thật sự lớn hơn của `p`) ⇒ **không** có dãy con tốt.
- Nếu `g = p`: lấy cả `M` là xong — **trừ khi** `M` là cả mảng (`|M| = n`), vì dãy con phải ngắn hơn `n`.

Tính tay ví dụ 1 sau truy vấn 2: `[3, 8, 6, 16]`, `p = 2` ⇒ `M = {8, 6, 16}`, `g = 2 = p`, `|M| = 3 < 4` ⇒ **có**.

Ví dụ 2 sau truy vấn 3: `[6, 9, 3, 8]`, `p = 3` ⇒ `M = {6, 9, 3}`, `g = 3` ⇒ có.

## 5. Cách 1 — Brute Force: thử mọi dãy con sau mỗi truy vấn

Với mỗi truy vấn, duyệt mọi tập con khác rỗng có kích thước `< n`, kiểm tra GCD. `O(q · 2^n · n)`.

In [5]:
def has_good_brute(nums, p):
    n = len(nums)
    for size in range(1, n):
        for comb in combinations(nums, size):
            if reduce(gcd, comb) == p:
                return True
    return False

def brute(nums, p, queries):
    a = nums[:]
    ans = 0
    for i, v in queries:
        a[i] = v
        ans += has_good_brute(a, p)
    return ans

for nums, p, qs, exp in examples:
    print(brute(nums, p, qs), "kỳ vọng", exp)

1 kỳ vọng 1
2 kỳ vọng 2
0 kỳ vọng 0


**Nút thắt:** `2^n` với `n = 5·10^4`. Cần dùng lập luận "chỉ GCD của các bội số quan trọng".

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề 1 (quy về GCD toàn cục của các bội).** Có dãy con tốt ⇔ `g = p` **và** (`|M| < n` **hoặc** có chỉ số `i` mà `gcd(M \ {a_i}) = p`).

- Nếu `|M| = n`, một dãy con tốt là tập con thực sự của mảng; nó nằm trong một tập `M \ {a_i}` nào đó, mà GCD của tập lớn hơn **chia hết** GCD của tập nhỏ hơn và đều là bội của `p` ⇒ chỉ cần thử bỏ **đúng một** phần tử.

**Bổ đề 2 (chỉ cần thử khi `n ≤ 6`).** Giả sử mọi phần tử là bội của `p`, `gcd = p`, và **với mọi `i`**, `g_i = gcd(các phần tử trừ a_i) > p`. Chia mọi thứ cho `p`: `b_i = a_i / p ≤ 5·10^4`, `h_i = g_i / p > 1`.

- `h_i` và `h_j` (`i ≠ j`) **nguyên tố cùng nhau**: ước chung của chúng chia hết mọi `b_k` (vì `h_i` chia hết mọi `b_k` với `k ≠ i`, còn `h_j` chia hết `b_i`) ⇒ chia hết `gcd = 1`.
- `b_i` chia hết cho tích `∏_{j ≠ i} h_j` (các số đôi một nguyên tố cùng nhau) ⇒ `b_i ≥ ∏_{j≠i} h_j`.
- Với `n` số `h` đôi một nguyên tố cùng nhau `> 1`, tích `P = ∏ h_j ≥ 2·3·5·…` (n số nguyên tố đầu). Lấy `i` có `h_i` **nhỏ nhất** ⇒ `b_i ≥ P / h_i`. Với `n = 7`: nếu `h_min ≤ 10` thì `P / h_min ≥` tích 6 số còn lại `≥ 11·13·17·19·23·29 ≫ 5·10^4`; nếu `h_min ≥ 11` thì mọi `h ≥ 11` và tích 6 số bất kỳ còn lớn hơn nữa. Mâu thuẫn ⇒ với `n ≥ 7` **luôn** bỏ được một phần tử.
- Với `n = 6` thì có thể xảy ra (ví dụ `h = 2,3,5,7,11,13`, `b_i = 30030 / h_i ≤ 15015`) ⇒ phải kiểm tra thật.

**Thuật toán:** duy trì `g` bằng segment tree (vị trí không là bội của `p` lưu `0`) và `cnt = |M|`. Sau mỗi truy vấn:

- `g ≠ p` ⇒ không.
- `cnt < n` hoặc `n ≥ 7` ⇒ có.
- Ngược lại (`n ≤ 6`, mọi phần tử là bội) ⇒ thử bỏ từng phần tử (`n ≤ 6` lần).

In [6]:
# Minh họa bổ đề 2: một bộ 6 số (đã chia cho p) có gcd = 1 nhưng bỏ bất kỳ phần tử nào gcd vẫn > 1
P = 2 * 3 * 5 * 7 * 11 * 13
b = [P // h for h in (2, 3, 5, 7, 11, 13)]
print("b =", b, "| max =", max(b), "<= 5*10^4:", max(b) <= 5 * 10**4)
print("gcd cả bộ:", reduce(gcd, b), "| gcd khi bỏ từng phần tử:", [reduce(gcd, b[:i] + b[i + 1:]) for i in range(6)])

b = [15015, 10010, 6006, 4290, 2730, 2310] | max = 15015 <= 5*10^4: True
gcd cả bộ: 1 | gcd khi bỏ từng phần tử: [2, 3, 5, 7, 11, 13]


## 7. Các cách tiếp theo

- **Cách 2:** tính lại `g`, `cnt` trong `O(n)` sau **mỗi** truy vấn, kiểm tra bỏ một phần tử bằng tiền tố/hậu tố — `O(n · q)`.
- **Cách 3:** **segment tree GCD** + đếm `cnt` + bổ đề `n ≤ 6` — `O((n + q) log n · log V)` (lời giải nộp chính).
- **Cách 4:** **chia căn**: chia mảng thành khối `√n`, mỗi khối lưu GCD; cập nhật tính lại một khối — `O(q √n log V)`.

### 7.1. Cách 2 — Tính lại toàn bộ sau mỗi truy vấn

Không dùng bổ đề 2: kiểm tra trực tiếp "bỏ một phần tử" bằng GCD tiền tố/hậu tố khi `cnt = n`.

In [7]:
def check_array(a, p):
    n = len(a)
    vals = [x if x % p == 0 else 0 for x in a]
    g = reduce(gcd, vals, 0)
    if g != p:
        return False
    cnt = sum(1 for x in a if x % p == 0)
    if cnt < n:
        return True
    pre = [0] * (n + 1); suf = [0] * (n + 1)
    for i in range(n): pre[i + 1] = gcd(pre[i], vals[i])
    for i in range(n - 1, -1, -1): suf[i] = gcd(suf[i + 1], vals[i])
    return any(gcd(pre[i], suf[i + 1]) == p for i in range(n))

def recompute(nums, p, queries):
    a = nums[:]
    ans = 0
    for i, v in queries:
        a[i] = v
        ans += check_array(a, p)
    return ans

for nums, p, qs, exp in examples:
    print(recompute(nums, p, qs), "kỳ vọng", exp)

1 kỳ vọng 1
2 kỳ vọng 2
0 kỳ vọng 0


**Độ phức tạp Cách 2:** `O(n · q · log V)` ≈ `2,5·10^9` — quá chậm, nhưng là "trọng tài" tốt cho test cỡ vừa (không phụ thuộc bổ đề 2).

### 7.2. Cách 3 — Segment tree GCD (lời giải chính)

**Giải thích từng đoạn:**

1. Khởi tạo lá `i` = `nums[i]` nếu chia hết cho `p`, ngược lại `0`; `cnt` = số bội.
2. Truy vấn `[i, v]`: điều chỉnh `cnt` theo giá trị cũ/mới, `update(i, v if v % p == 0 else 0)`.
3. `g = t[1]`. Nếu `g != p` → bỏ qua. Nếu `cnt < n` hoặc `n > 6` → đếm. Ngược lại thử `gcd(query(0, i), query(i+1, n)) == p` cho từng `i`.

In [8]:
def seg_solution(nums, p, queries):
    n = len(nums)
    a = nums[:]
    st = SegGCD([x if x % p == 0 else 0 for x in a])
    cnt = sum(1 for x in a if x % p == 0)
    ans = 0
    for i, v in queries:
        cnt -= a[i] % p == 0
        cnt += v % p == 0
        a[i] = v
        st.update(i, v if v % p == 0 else 0)
        if st.total() != p:
            continue
        if cnt < n or n > 6:
            ans += 1
            continue
        if any(gcd(st.query(0, j), st.query(j + 1, n)) == p for j in range(n)):
            ans += 1
    return ans

for nums, p, qs, exp in examples:
    print(seg_solution(nums, p, qs), "kỳ vọng", exp)

1 kỳ vọng 1
2 kỳ vọng 2
0 kỳ vọng 0


**Độ phức tạp Cách 3:** khởi tạo `O(n)`; mỗi truy vấn `O(log n)` lần `gcd` (mỗi lần `O(log V)`), cộng tối đa `6` truy vấn đoạn khi `n ≤ 6`. Tổng `O((n + q) log n log V)`; bộ nhớ `O(n)`.

### 7.3. Cách 4 — Chia căn (sqrt decomposition)

Chia mảng thành khối kích thước `B ≈ √n`; `block_g[b]` = GCD các giá trị (đã lọc) trong khối. Cập nhật: tính lại GCD **một khối** `O(B log V)`. GCD toàn cục: GCD của `n / B` khối `O(√n log V)`.

In [9]:
def sqrt_solution(nums, p, queries):
    n = len(nums)
    a = nums[:]
    vals = [x if x % p == 0 else 0 for x in a]
    B = max(1, int(n ** 0.5))
    nb = (n + B - 1) // B
    block_g = [reduce(gcd, vals[b * B:(b + 1) * B], 0) for b in range(nb)]
    cnt = sum(1 for x in a if x % p == 0)
    ans = 0
    for i, v in queries:
        cnt -= a[i] % p == 0
        cnt += v % p == 0
        a[i] = v
        vals[i] = v if v % p == 0 else 0
        b = i // B
        block_g[b] = reduce(gcd, vals[b * B:(b + 1) * B], 0)
        if reduce(gcd, block_g, 0) != p:
            continue
        if cnt < n or n > 6:
            ans += 1
            continue
        if any(reduce(gcd, vals[:j] + vals[j + 1:], 0) == p for j in range(n)):
            ans += 1
    return ans

for nums, p, qs, exp in examples:
    print(sqrt_solution(nums, p, qs), "kỳ vọng", exp)

1 kỳ vọng 1
2 kỳ vọng 2
0 kỳ vọng 0


**Độ phức tạp Cách 4:** `O(q · √n · log V)` ≈ `5·10^4 · 224 ≈ 10^7` phép `gcd` — chấp nhận được nhưng chậm hơn segment tree.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 2: `nums = [4,5,7,8]`, `p = 3`. Ban đầu không có bội của 3 ⇒ lá `[0,0,0,0]`, `cnt = 0`.

| truy vấn | mảng sau cập nhật | lá (bội của 3) | g = t[1] | cnt | kết luận |
|---|---|---|---|---|---|
| [0, 6] | [6,5,7,8] | [6,0,0,0] | 6 | 1 | g ≠ 3 ⇒ không |
| [1, 9] | [6,9,7,8] | [6,9,0,0] | 3 | 2 | g = 3, cnt < n ⇒ **có** |
| [2, 3] | [6,9,3,8] | [6,9,3,0] | 3 | 3 | **có** |

In [10]:
def trace_seg(nums, p, queries):
    n = len(nums); a = nums[:]
    st = SegGCD([x if x % p == 0 else 0 for x in a])
    cnt = sum(1 for x in a if x % p == 0); ans = 0
    print("ban đầu lá =", st.t[st.size:st.size + n], "cnt =", cnt)
    for i, v in queries:
        cnt += (v % p == 0) - (a[i] % p == 0); a[i] = v
        st.update(i, v if v % p == 0 else 0)
        g = st.total()
        if g != p:
            verdict = "g != p -> không"
        elif cnt < n or n > 6:
            verdict = "g == p, cnt < n hoặc n > 6 -> có"; ans += 1
        else:
            ok = any(gcd(st.query(0, j), st.query(j + 1, n)) == p for j in range(n))
            verdict = f"g == p, cnt == n <= 6 -> thử bỏ 1 phần tử -> {'có' if ok else 'không'}"; ans += ok
        print(f"[{i},{v}] mảng={a} lá={st.t[st.size:st.size + n]} g={g} cnt={cnt} | {verdict}")
    print("đáp án =", ans)

trace_seg([4, 5, 7, 8], 3, [[0, 6], [1, 9], [2, 3]])

ban đầu lá = [0, 0, 0, 0] cnt = 0
[0,6] mảng=[6, 5, 7, 8] lá=[6, 0, 0, 0] g=6 cnt=1 | g != p -> không
[1,9] mảng=[6, 9, 7, 8] lá=[6, 9, 0, 0] g=3 cnt=2 | g == p, cnt < n hoặc n > 6 -> có
[2,3] mảng=[6, 9, 3, 8] lá=[6, 9, 3, 0] g=3 cnt=3 | g == p, cnt < n hoặc n > 6 -> có
đáp án = 2


In [11]:
# Trace trường hợp cnt == n <= 6: mọi phần tử là bội của p
trace_seg([6, 10, 15], 1, [[0, 6], [2, 30]])

ban đầu lá = [6, 10, 15] cnt = 3
[0,6] mảng=[6, 10, 15] lá=[6, 10, 15] g=1 cnt=3 | g == p, cnt == n <= 6 -> thử bỏ 1 phần tử -> không
[2,30] mảng=[6, 10, 30] lá=[6, 10, 30] g=2 cnt=3 | g != p -> không
đáp án = 0


Ở truy vấn đầu, `[6, 10, 15]` có GCD 1 nhưng bỏ bất kỳ phần tử nào GCD còn lại là `5`, `3`, `2` ⇒ **không** có dãy con tốt (phải ngắn hơn 3 phần tử). Sau khi đổi `15 → 30`: `[6, 10, 30]` có GCD 2 ≠ 1 ⇒ không.

## 9. Code nộp LeetCode

In [12]:
from math import gcd

class Solution:
    def countGoodSubseq(self, nums: list[int], p: int, queries: list[list[int]]) -> int:
        n = len(nums)
        size = 1
        while size < n:
            size *= 2
        t = [0] * (2 * size)
        cnt = 0
        for i, x in enumerate(nums):
            if x % p == 0:
                t[size + i] = x
                cnt += 1
        for v in range(size - 1, 0, -1):
            t[v] = gcd(t[2 * v], t[2 * v + 1])

        def query(l, r):                # gcd trên [l, r)
            res = 0
            l += size; r += size
            while l < r:
                if l & 1:
                    res = gcd(res, t[l]); l += 1
                if r & 1:
                    r -= 1; res = gcd(res, t[r])
                l >>= 1; r >>= 1
            return res

        ans = 0
        for i, val in queries:
            if nums[i] % p == 0:
                cnt -= 1
            if val % p == 0:
                cnt += 1
            nums[i] = val
            v = size + i
            t[v] = val if val % p == 0 else 0
            v >>= 1
            while v:
                t[v] = gcd(t[2 * v], t[2 * v + 1])
                v >>= 1
            if t[1] != p:
                continue
            if cnt < n or n > 6:
                ans += 1
            elif any(gcd(query(0, j), query(j + 1, n)) == p for j in range(n)):
                ans += 1
        return ans

sol = Solution()
for nums, p, qs, exp in examples:
    got = sol.countGoodSubseq(nums[:], p, qs)
    print(got, "✔" if got == exp else "✘")

1 ✔
2 ✔
0 ✔


## 10. Vì sao thuật toán đúng

1. **Chỉ bội của `p` tham gia:** GCD của dãy con bằng `p` ⇒ `p` chia hết mọi phần tử.
2. **`g ≠ p` ⇒ không:** mọi dãy con các bội của `p` có GCD là bội của `g`. Nếu `M` rỗng thì không có dãy con nào. Nếu `g` là bội thực sự của `p` (`g > p`) thì GCD mọi dãy con `≥ g > p`.
3. **`g = p`, `cnt < n` ⇒ có:** chính `M` là dãy con độ dài `cnt < n` có GCD `p`.
4. **`g = p`, `cnt = n`:** dãy con tốt `S` (độ dài `< n`) thiếu ít nhất một chỉ số `i` ⇒ `S ⊆ M \ {a_i}` ⇒ `p | gcd(M \ {a_i}) | gcd(S) = p` ⇒ `gcd(M \ {a_i}) = p`. Nên chỉ cần kiểm tra bỏ đúng một phần tử. Với `n ≥ 7`, bổ đề 2 (mục 6) cho biết luôn có `i` như vậy; với `n ≤ 6` ta thử trực tiếp.
5. **Segment tree** duy trì đúng GCD (với `0` là phần tử trung hòa) sau mỗi cập nhật điểm.

## 11. Phân tích Time / Space Complexity

Đặt `n` = độ dài, `q` = số truy vấn, `V = 5·10^4` (chi phí `gcd` là `O(log V)`).

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(q · 2^n · n)` | `O(n)` |
| 2. Tính lại mỗi truy vấn | `O(q · n · log V)` | `O(n)` |
| 3. Segment tree | `O((n + q) · log n · log V)` | `O(n)` |
| 4. Chia căn | `O(q · √n · log V)` | `O(n)` |

## 12. Bộ test

In [13]:
def run_all(nums, p, qs, with_brute=True):
    res = [recompute(nums, p, qs), seg_solution(nums, p, qs), sqrt_solution(nums, p, qs),
           Solution().countGoodSubseq(nums[:], p, qs)]
    if with_brute:
        res.append(brute(nums, p, qs))
    return res

for nums, p, qs, exp in examples:
    assert all(v == exp for v in run_all(nums, p, qs)), run_all(nums, p, qs)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [14]:
# 12.2 ca biên (tự dựng, kiểm chứng bằng vét cạn)
cases = [
    ([2, 2], 2, [[0, 2]]),                         # n = 2: dãy con [2] có gcd 2 -> có
    ([2, 4], 2, [[1, 2]]),                         # [2,2] -> có
    ([6, 10, 15], 1, [[1, 10]]),                   # gcd 1 nhưng bỏ 1 phần tử luôn > 1 -> không
    ([1, 1, 1], 1, [[0, 1], [2, 5]]),              # có
    ([30030 // h for h in (2, 3, 5, 7, 11, 13)], 1, [[0, 15015]]),   # n = 6 phản ví dụ cho "n > 5"
    ([5, 5], 7, [[0, 14]]),                         # [14,5]: g = 14 != 7 -> không
]
for nums, p, qs in cases:
    r = run_all(nums, p, qs)
    assert len(set(r)) == 1, (nums, p, qs, r)
    print(nums, p, qs, "->", r[0])

[2, 2] 2 [[0, 2]] -> 1
[2, 4] 2 [[1, 2]] -> 1
[6, 10, 15] 1 [[1, 10]] -> 0
[1, 1, 1] 1 [[0, 1], [2, 5]] -> 2
[15015, 10010, 6006, 4290, 2730, 2310] 1 [[0, 15015]] -> 0
[5, 5] 7 [[0, 14]] -> 0


In [15]:
# 12.3 phản ví dụ: nếu dùng ngưỡng sai "n > 5 thì luôn có" sẽ sai ở bộ 6 số đặc biệt
def wrong_threshold(nums, p, queries):
    a = nums[:]; n = len(a); ans = 0
    for i, v in queries:
        a[i] = v
        vals = [x if x % p == 0 else 0 for x in a]
        if reduce(gcd, vals, 0) != p: continue
        cnt = sum(1 for x in a if x % p == 0)
        if cnt < n or n > 5:                     # SAI: ngưỡng phải là n > 6
            ans += 1; continue
        ans += any(reduce(gcd, vals[:j] + vals[j + 1:], 0) == p for j in range(n))
    return ans

nums6 = [30030 // h for h in (2, 3, 5, 7, 11, 13)]
print("sai:", wrong_threshold(nums6, 1, [[0, 15015]]), "| đúng:", Solution().countGoodSubseq(nums6[:], 1, [[0, 15015]]))

sai: 1 | đúng: 0


In [16]:
# 12.4 đối chiếu ngẫu nhiên (n nhỏ để vét cạn được)
rng = random.Random(3901)
for _ in range(250):
    n = rng.randint(2, 7)
    p = rng.choice([1, 2, 3, 4, 6])
    nums = [p * rng.randint(1, 12) if rng.random() < 0.7 else rng.randint(1, 60) for _ in range(n)]
    qs = [[rng.randrange(n), p * rng.randint(1, 12) if rng.random() < 0.7 else rng.randint(1, 60)] for _ in range(rng.randint(1, 6))]
    r = run_all(nums, p, qs)
    assert len(set(r)) == 1, (nums, p, qs, r)
print("250 test ngẫu nhiên (có vét cạn): OK")
for _ in range(60):
    n = rng.randint(7, 60); p = rng.randint(1, 5)
    nums = [p * rng.randint(1, 30) for _ in range(n)]
    qs = [[rng.randrange(n), p * rng.randint(1, 30)] for _ in range(30)]
    r = run_all(nums, p, qs, with_brute=False)
    assert len(set(r)) == 1, r
print("60 test n >= 7 (mọi phần tử là bội, so với Cách 2 không dùng bổ đề): OK")

250 test ngẫu nhiên (có vét cạn): OK
60 test n >= 7 (mọi phần tử là bội, so với Cách 2 không dùng bổ đề): OK


In [17]:
# 12.5 kiểm chứng bổ đề 2 bằng tìm kiếm: không tồn tại bộ 7 số <= 5*10^4 gcd 1 mà bỏ phần tử nào gcd cũng > 1
# (kiểm tra ngẫu nhiên có định hướng: sinh từ các bộ h đôi một nguyên tố cùng nhau nhỏ nhất)
from itertools import combinations as C
primes = [2, 3, 5, 7, 11, 13, 17, 19, 23]
best = min(max(__import__('math').prod(hs) // h for h in hs) for hs in C(primes, 7))
print("với 7 số nguyên tố bất kỳ trong 9 số đầu, max b_i nhỏ nhất có thể =", best, "> 5*10^4:", best > 5 * 10**4)

# test lớn đúng ràng buộc
rng = random.Random(9)
n = 5 * 10**4
nums = [rng.randint(1, 5 * 10**4) for _ in range(n)]
qs = [[rng.randrange(n), rng.choice([2, 4, 6, rng.randint(1, 5 * 10**4)])] for _ in range(5 * 10**4)]
t0 = time.time(); a = Solution().countGoodSubseq(nums[:], 2, qs); t1 = time.time()
print("đáp án:", a, "| thời gian < 3s:", t1 - t0 < 3)

với 7 số nguyên tố bất kỳ trong 9 số đầu, max b_i nhỏ nhất có thể = 255255 > 5*10^4: True


đáp án: 50000 | thời gian < 3s: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tính GCD của **cả mảng** (gồm cả số không chia hết cho `p`)

In [18]:
def wrong_all_gcd(nums, p, queries):
    a = nums[:]; ans = 0
    for i, v in queries:
        a[i] = v
        if reduce(gcd, a) == p:                  # SAI: số không chia hết cho p làm gcd nhỏ đi
            ans += 1
    return ans

nums, p, qs = [4, 8, 12, 16], 2, [[0, 3], [2, 6]]
print("sai:", wrong_all_gcd(nums, p, qs), "| đúng:", Solution().countGoodSubseq(nums[:], p, qs))

sai: 0 | đúng: 1


**Vì sao sai:** sau truy vấn 2 mảng `[3, 8, 6, 16]` có GCD 1 ≠ 2, nhưng dãy con `[8, 6]` vẫn có GCD 2. **Sửa:** bỏ (gán 0) các số không chia hết cho `p`.

### Lỗi 2 — Quên điều kiện "độ dài nhỏ hơn `n`"

In [19]:
def wrong_no_length(nums, p, queries):
    a = nums[:]; ans = 0
    for i, v in queries:
        a[i] = v
        if reduce(gcd, [x if x % p == 0 else 0 for x in a], 0) == p:
            ans += 1                              # SAI: chưa xét trường hợp phải bỏ phần tử
    return ans

print("sai:", wrong_no_length([6, 10, 15], 1, [[1, 10]]), "| đúng:", Solution().countGoodSubseq([6, 10, 15], 1, [[1, 10]]))

sai: 1 | đúng: 0


### Lỗi 3 — Cập nhật `cnt` sai thứ tự (gán `nums[i]` trước khi trừ giá trị cũ)

In [20]:
def wrong_cnt_order(nums, p, queries):
    a = nums[:]; n = len(a); cnt = sum(1 for x in a if x % p == 0); ans = 0
    for i, v in queries:
        a[i] = v                                  # SAI: ghi đè trước
        cnt -= a[i] % p == 0                      # đang trừ theo giá trị MỚI
        cnt += v % p == 0
        vals = [x if x % p == 0 else 0 for x in a]
        if reduce(gcd, vals, 0) == p and (cnt < n or any(reduce(gcd, vals[:j] + vals[j+1:], 0) == p for j in range(n))):
            ans += 1
    return ans, cnt

print("sai (ans, cnt):", wrong_cnt_order([3, 5], 3, [[1, 6]]), "| đúng ans:", Solution().countGoodSubseq([3, 5], 3, [[1, 6]]))

sai (ans, cnt): (1, 1) | đúng ans: 1


**Vì sao sai:** `cnt` không tăng khi `5 → 6`, nên `cnt = 1 < n = 2` dù thực ra cả mảng là bội của 3 — khiến thuật toán trả "có" mà không kiểm tra bỏ phần tử. Ở ví dụ này kết quả trùng hợp vẫn đúng (`[3]` hoặc `[6]`… GCD 3), nhưng `cnt` đã sai; với dữ liệu khác sẽ ra đáp án sai. **Sửa:** trừ theo giá trị cũ **trước** khi gán.

In [21]:
# Dữ liệu làm lộ lỗi 3: mảng toàn bội, gcd = p nhưng không bỏ được phần tử nào
print("sai:", wrong_cnt_order([12, 20, 7], 2, [[2, 30]])[0], "| đúng:", Solution().countGoodSubseq([12, 20, 7], 2, [[2, 30]]))
# [12, 20, 30]: gcd 2 nhưng bỏ phần tử nào cũng ra 10, 6, 4 -> không có dãy con tốt

sai: 1 | đúng: 0


### Lỗi 4 — Thử bỏ phần tử bằng cách tạo mảng mới mỗi lần với `n` lớn

`reduce(gcd, vals[:j] + vals[j+1:])` cho mọi `j` là `O(n²)` mỗi truy vấn. Với `n = 5·10^4` là không thể. Hoặc dùng bổ đề 2 (chỉ làm khi `n ≤ 6`), hoặc dùng segment tree/tiền tố–hậu tố.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"GCD đúng bằng `p`"** ⇒ lọc bội của `p`, rồi xét **GCD của tất cả** bội (GCD lớn nhất có thể đạt được theo nghĩa chia hết).
- **GCD + cập nhật điểm** ⇒ segment tree (GCD không có phép "trừ" như tổng, nên Fenwick không dùng được).
- **"Bỏ đúng một phần tử"** ⇒ GCD tiền tố/hậu tố hoặc truy vấn hai đoạn trên segment tree.
- Ràng buộc giá trị nhỏ (`≤ 5·10^4`) thường gợi ý một **bổ đề số học** giới hạn kích thước trường hợp khó (ở đây `n ≤ 6`).

## 15. Biến thể của bài

1. **Đếm số dãy con có GCD = `p`** (không truy vấn): Möbius / bao hàm–loại trừ trên bội số: `f(d) = 2^{c(d)} - 1` với `c(d)` = số phần tử chia hết cho `d`, rồi `exact(d) = f(d) - Σ_{k≥2} exact(kd)`.
2. **Dãy con liên tiếp (subarray)** có GCD = `p`: duy trì tập GCD của các subarray kết thúc tại `i` (tối đa `log V` giá trị).
3. **Không có điều kiện độ dài**: bỏ phần kiểm tra `cnt == n`.
4. **Truy vấn đoạn `[l, r]`**: segment tree trả GCD đoạn các bội của `p`.

In [22]:
# Biến thể 1: đếm số dãy con (khác rỗng) có GCD đúng bằng p, đối chiếu vét cạn
def count_exact_gcd(nums, p):
    MOD = 10**9 + 7
    V = max(nums)
    freq = [0] * (V + 1)
    for x in nums: freq[x] += 1
    exact = [0] * (V + 1)
    for d in range(V, 0, -1):
        c = sum(freq[m] for m in range(d, V + 1, d))
        exact[d] = (pow(2, c, MOD) - 1 - sum(exact[m] for m in range(2 * d, V + 1, d))) % MOD
    return exact[p] if p <= V else 0

def brute_count(nums, p):
    return sum(1 for k in range(1, len(nums) + 1) for c in combinations(nums, k) if reduce(gcd, c) == p)

rng = random.Random(15)
for _ in range(200):
    nums = [rng.randint(1, 24) for _ in range(rng.randint(1, 9))]
    p = rng.randint(1, 6)
    assert count_exact_gcd(nums, p) == brute_count(nums, p)
print("Biến thể 1 (đếm bằng bao hàm–loại trừ) khớp vét cạn.")

Biến thể 1 (đếm bằng bao hàm–loại trừ) khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Thử mọi dãy con | `O(q 2^n n)` | `O(n)` | Kiểm thử |
| 2. Tính lại | GCD bội + tiền/hậu tố mỗi truy vấn | `O(q n log V)` | `O(n)` | `n·q ≤ 10^7`, trọng tài |
| 3. Segment tree | GCD toàn cục cập nhật điểm + bổ đề `n ≤ 6` | `O((n+q) log n log V)` | `O(n)` | **Lời giải chuẩn** |
| 4. Chia căn | GCD theo khối `√n` | `O(q √n log V)` | `O(n)` | Khi không muốn viết segment tree |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `nums = [3, 3], p = 3, queries = [[0, 6]]` → ?

**Bài 2 (tính tay).** `nums = [10, 15, 6, 7], p = 1, queries = [[3, 30], [3, 1]]` — tính từng truy vấn.

**Bài 3 (điền TODO).** Viết `can_drop_one(vals, p)` dùng GCD tiền tố/hậu tố.

**Bài 4 (tự viết lại).** Viết lại lời giải chính **không dùng bổ đề 2**: luôn kiểm tra bỏ một phần tử, nhưng chỉ cần `O(log n)` mỗi truy vấn. (Gợi ý: phần tử "cần thiết" là phần tử mà bỏ nó thì GCD tăng; dùng segment tree để thử… hoặc nhận xét số phần tử cần thử bị chặn.)

**Bài 5 (mini challenge).** Hỏi thêm: sau mỗi truy vấn, **độ dài nhỏ nhất** của dãy con tốt là bao nhiêu (hoặc `-1`)? Viết bản `O(n · 2^n)` cho `n ≤ 10` để kiểm chứng.

In [23]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def can_drop_one(vals, p):
    # TODO: pre/suf gcd, trả True nếu có i với gcd(pre[i], suf[i+1]) == p
    pass

In [24]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def min_good_length(nums, p):
    # TODO: duyệt mọi tập con kích thước < n, lấy độ dài nhỏ nhất có gcd == p
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `[6, 3]`, bội của 3: cả hai. (2) `g = 3`, `cnt = n = 2`. (3) Bỏ `6` còn `[3]` GCD 3 ⇒ có ⇒ `1`.

**Bài 2.** (1) Truy vấn 1: `[10,15,6,30]`, `g = 1`, `cnt = n = 4 ≤ 6` ⇒ thử bỏ: bỏ `30` còn `gcd(10,15,6) = 1` ⇒ có. (2) Truy vấn 2: `[10,15,6,1]` ⇒ `[1]` là dãy con tốt ⇒ có. (3) Đáp án `2`.

**Bài 3.** (1) `pre[0] = 0`. (2) `suf[n] = 0`. (3) `gcd(0, x) = x`.

**Bài 4.** (1) Theo bổ đề, chỉ cần thử khi `n ≤ 6`, nên "luôn thử" với `n ≤ 6` là `O(6 log n)`. (2) Khi `n ≥ 7` đáp án chắc chắn có — có thể kiểm chứng bằng Cách 2. (3) So kết quả ngẫu nhiên.

**Bài 5.** (1) Duyệt kích thước tăng dần. (2) Dừng ở kích thước đầu tiên tìm thấy. (3) Nếu `g ≠ p` thì `-1` ngay.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1, 2
assert Solution().countGoodSubseq([3, 3], 3, [[0, 6]]) == 1 == brute([3, 3], 3, [[0, 6]])
assert Solution().countGoodSubseq([10, 15, 6, 7], 1, [[3, 30], [3, 1]]) == 2 == brute([10, 15, 6, 7], 1, [[3, 30], [3, 1]])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [26]:
# Bài 3
def can_drop_one_sol(vals, p):
    n = len(vals)
    pre = [0] * (n + 1); suf = [0] * (n + 1)
    for i in range(n): pre[i + 1] = gcd(pre[i], vals[i])
    for i in range(n - 1, -1, -1): suf[i] = gcd(suf[i + 1], vals[i])
    return any(gcd(pre[i], suf[i + 1]) == p for i in range(n))

assert can_drop_one_sol([6, 10, 15], 1) is False
assert can_drop_one_sol([6, 10, 15, 1], 1) is True
assert can_drop_one_sol([4, 6, 8], 2) is True
print("Bài 3 OK")

Bài 3 OK


In [27]:
# Bài 4: luôn kiểm tra bỏ một phần tử khi n <= 6, và với n >= 7 dùng kết luận của bổ đề — đối chiếu Cách 2
def sol_always_check(nums, p, queries):
    n = len(nums); a = nums[:]; ans = 0
    for i, v in queries:
        a[i] = v
        vals = [x if x % p == 0 else 0 for x in a]
        if reduce(gcd, vals, 0) != p: continue
        cnt = sum(1 for x in a if x % p == 0)
        ans += cnt < n or can_drop_one_sol(vals, p)
    return ans

rng = random.Random(4)
for _ in range(150):
    n = rng.randint(2, 30); p = rng.randint(1, 4)
    nums = [p * rng.randint(1, 20) for _ in range(n)]
    qs = [[rng.randrange(n), p * rng.randint(1, 20)] for _ in range(10)]
    assert sol_always_check(nums, p, qs) == Solution().countGoodSubseq(nums[:], p, qs)
print("Bài 4 OK")

Bài 4 OK


In [28]:
# Bài 5
def min_good_length_sol(nums, p):
    n = len(nums)
    for size in range(1, n):
        for c in combinations(nums, size):
            if reduce(gcd, c) == p:
                return size
    return -1

assert min_good_length_sol([3, 8, 6, 16], 2) == 2
assert min_good_length_sol([6, 9, 3, 8], 3) == 1
assert min_good_length_sol([6, 10, 15], 1) == -1
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# lá = x nếu x % p == 0 else 0;  cnt = số bội của p
for i, val in queries:
    cnt += (val % p == 0) - (nums[i] % p == 0)   # trừ theo giá trị CŨ trước khi gán
    nums[i] = val; update(i, val if val % p == 0 else 0)
    if t[1] != p: continue
    if cnt < n or n > 6: ans += 1
    elif any(gcd(query(0, j), query(j+1, n)) == p for j in range(n)): ans += 1
```

**Checklist:**

- [ ] Chỉ tính GCD trên các **bội của `p`** (vị trí khác lưu `0`)?
- [ ] `g == p` chứ không phải `g % p == 0`?
- [ ] Xử lý điều kiện **độ dài < n** (bỏ một phần tử khi `cnt == n`)?
- [ ] Ngưỡng bổ đề `n > 6` (không phải `n > 5`)?
- [ ] Cập nhật `cnt` theo giá trị cũ **trước** khi gán giá trị mới?